# 09 - Model Governance and Monitoring

This notebook converts the modelling, threshold-selection, and explainability outputs into governance-ready documentation.

The goal is to show how a Canadian retail credit-risk analytics project would be controlled after model development:

- model inventory and intended use
- validation/test performance summary
- leakage and fairness/proxy controls
- explainability governance
- operating threshold controls
- monitoring KPIs and escalation limits
- model card, validation summary, stakeholder brief, and monitoring plan

## Professional framing

A credit-risk model is not complete when the ROC-AUC or PR-AUC is calculated. For a banking-style portfolio project, the model must also be explainable, monitored, and governed.

This notebook positions the model as a **decision-support and manual-review prioritization tool**, not an automated credit-decline engine. That distinction is important because the dataset and project do not include all production controls required for automated lending decisions.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from credit_risk.config import GOVERNANCE_DIR, PROCESSED_DIR, TABLE_DIR, ensure_project_directories
from credit_risk.governance.model_governance import (
    build_control_register,
    build_feature_governance_summary,
    build_governance_summary,
    build_model_inventory,
    build_model_risk_limit_register,
    build_monitoring_kpi_snapshot,
    build_validation_test_summary,
    build_xai_governance_summary,
    load_governance_inputs,
    save_governance_outputs,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)
ensure_project_directories()

## Load governance inputs

This step reads outputs produced by earlier notebooks:

- modelling dataset and feature catalogue from Notebook 05
- model results from Notebook 06
- threshold outputs from Notebook 07
- XAI outputs from Notebook 08

In [ ]:
inputs = load_governance_inputs(processed_dir=PROCESSED_DIR, table_dir=TABLE_DIR)

{
    "modeling_rows": len(inputs.modeling_df),
    "modeling_columns": inputs.modeling_df.shape[1],
    "feature_catalog_rows": len(inputs.feature_catalog),
    "model_selection_rows": len(inputs.model_selection),
    "threshold_summary_rows": len(inputs.recommended_threshold),
    "xai_grouped_importance_rows": len(inputs.xai_grouped_importance),
}

## Model inventory

The inventory table defines what the model is, what it is used for, and the key governance boundaries.

In [ ]:
model_inventory = build_model_inventory(inputs)
model_inventory

## Validation, test, and operating-threshold summary

The table compares the original default 0.50 model output with the selected business operating threshold. The selected threshold is the relevant view for operational review planning.

In [ ]:
validation_test_summary = build_validation_test_summary(inputs)
validation_test_summary

## Feature governance summary

This summarizes which variables are used, excluded, or retained only for monitoring/governance review. The most important control is that repayment-derived variables are not used as model predictors.

In [ ]:
feature_governance_summary = build_feature_governance_summary(inputs)
feature_governance_summary

## Explainability governance

The XAI summary converts SHAP drivers into governance notes. Data-quality and pricing-related drivers are useful but require careful interpretation.

In [ ]:
xai_governance_summary = build_xai_governance_summary(inputs, top_n=12)
xai_governance_summary

## Control register

This table is the project's model-risk control register. It links each risk to a control, evidence source, owner, and review cadence.

In [ ]:
control_register = build_control_register()
control_register

## Monitoring KPI snapshot

This is the first monitoring baseline for the model. In a real deployment, the same KPIs would be refreshed every month or after each material data refresh.

In [ ]:
monitoring_kpi_snapshot = build_monitoring_kpi_snapshot(inputs)
monitoring_kpi_snapshot

## Model risk limits and escalation triggers

The limits below are practical monitoring thresholds for a portfolio project. They are intentionally labelled as monitoring triggers, not regulatory limits.

In [ ]:
risk_limit_register = build_model_risk_limit_register(inputs)
risk_limit_register

## One-row governance summary

This table is suitable for the README, stakeholder deck, or interview explanation.

In [ ]:
governance_summary = build_governance_summary(inputs)
governance_summary

## Save governance tables and documentation

This step creates CSV tables plus markdown documents under `reports/governance/`.

In [ ]:
saved_outputs = save_governance_outputs(
    inputs=inputs,
    table_dir=TABLE_DIR,
    governance_dir=GOVERNANCE_DIR,
)

saved_outputs

## Preview stakeholder brief

The stakeholder brief is the non-technical explanation you can show in GitHub or discuss during interviews.

In [ ]:
stakeholder_brief_path = GOVERNANCE_DIR / "stakeholder_brief.md"
print(stakeholder_brief_path.read_text(encoding="utf-8")[:2500])

## Notebook 09 conclusions

Carry these decisions into the final README and portfolio summary:

1. The model is positioned as a **manual-review prioritization tool**, not an automated decline engine.
2. The selected threshold balances default capture and review-team workload.
3. Leakage controls exclude repayment-derived predictors from model training.
4. Sensitive/proxy-sensitive variables are excluded from the baseline model.
5. SHAP, anchor-like rules, and counterfactual diagnostics provide explainability evidence.
6. The project now includes a model card, validation summary, stakeholder brief, monitoring plan, control register, and risk-limit register.